# Hospedando Servidor MCP no Amazon Bedrock AgentCore Runtime - Autenticação de Entrada OAuth

## Visão Geral

Neste tutorial aprenderemos como hospedar servidores MCP (Model Context Protocol) no Amazon Bedrock AgentCore Runtime. Usaremos o SDK Python do Amazon Bedrock AgentCore para encapsular ferramentas MCP como um servidor MCP compatível com o Amazon Bedrock AgentCore.

O SDK Python do Amazon Bedrock AgentCore cuida dos detalhes de implementação do servidor MCP para que você possa se concentrar na funcionalidade principal das suas ferramentas. Ele transforma seu código nos contratos padronizados do protocolo MCP do AgentCore para comunicação direta.

### Detalhes do Tutorial

| Informação          | Detalhes                                                  |
|:--------------------|:----------------------------------------------------------|
| Tipo de tutorial    | Hospedagem de Ferramentas                                 |
| Tipo de ferramenta  | Servidor MCP                                              |
| Componentes         | Hospedagem de servidor MCP no AgentCore Runtime           |
| Vertical            | Cross-vertical                                            |
| Complexidade        | Fácil                                                     |
| SDK usado           | SDK Python do Amazon BedrockAgentCore e MCP               |

### Arquitetura do Tutorial

Neste tutorial descreveremos como implantar um servidor MCP no AgentCore runtime.

Para fins de demonstração, usaremos um servidor MCP simples com 3 ferramentas: `add_numbers`, `multiply_numbers` e `greet_user`

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### Recursos Principais do Tutorial

* Criação de servidores MCP com ferramentas personalizadas
* Teste de servidores MCP localmente
* Hospedagem de servidores MCP no Amazon Bedrock AgentCore Runtime
* Invocação de servidores MCP implantados com autenticação


## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas
* SDK do Amazon Bedrock AgentCore
* Biblioteca MCP (Model Context Protocol)
* Docker em execução

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

ssm_client = boto_session.client('ssm', region_name=region)
secrets_client = boto_session.client('secretsmanager', region_name=region)
agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)

tool_name = "mcp_server_agentcore"

## Entendendo MCP (Model Context Protocol)

MCP é um protocolo que permite que modelos de IA acessem com segurança dados externos e ferramentas. Conceitos principais:

* **Ferramentas**: Funções que a IA pode chamar para executar ações
* **Streamable HTTP**: Protocolo de transporte usado pelo AgentCore Runtime
* **Isolamento de Sessão**: Cada cliente recebe sessões isoladas via cabeçalho `Mcp-Session-Id`
* **Operação Stateless**: Servidores devem suportar operação stateless para escalabilidade

O AgentCore Runtime espera que servidores MCP sejam hospedados em `0.0.0.0:8000/mcp` como o caminho padrão.

### Estrutura do Projeto

Vamos configurar nosso projeto com a estrutura adequada:

```
mcp_server_project/
├── mcp_server.py              # Código principal do servidor MCP
├── my_mcp_client.py          # Cliente de teste local
├── my_mcp_client_remote.py   # Cliente de teste remoto
├── requirements.txt          # Dependências
└── __init__.py              # Marcador de pacote Python
```

## Criando Servidor MCP

Agora vamos criar nosso servidor MCP com três ferramentas simples. O servidor usa FastMCP com `stateless_http=True`, que é necessário para compatibilidade com o AgentCore Runtime.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### O que Este Código Faz

* **FastMCP**: Cria um servidor MCP que pode hospedar suas ferramentas
* **@mcp.tool()**: Decorador que transforma suas funções Python em ferramentas MCP
* **stateless_http=True**: Necessário para compatibilidade com o AgentCore Runtime
* **Ferramentas**: Três ferramentas simples demonstrando diferentes tipos de operações

## Criando Cliente de Teste Local

Antes de implantar no AgentCore Runtime, vamos criar um cliente para testar nosso servidor MCP localmente:

In [ ]:
%%writefile my_mcp_client.py
import asyncio
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

### Testando Localmente

Para testar seu servidor MCP localmente:

1. **Terminal 1**: Inicie o servidor MCP
   ```bash
   python mcp_server.py
   ```
   
2. **Terminal 2**: Execute o cliente de teste
   ```bash
   python my_mcp_client.py
   ```

Você deve ver suas três ferramentas listadas na saída.

## Configurando o Amazon Cognito para Autenticação

O AgentCore Runtime requer autenticação. Usaremos o Amazon Cognito para fornecer tokens JWT para acessar nosso servidor MCP implantado.

In [ ]:
import sys
sys.path.insert(0, '../..')
from utils import setup_cognito_user_pool

print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## Configurando Implantação do AgentCore Runtime

Em seguida, usaremos nosso kit inicial para configurar a implantação do AgentCore Runtime com um entrypoint, a role de execução que acabamos de criar e um arquivo de requisitos. Também configuraremos o kit inicial para criar automaticamente o repositório Amazon ECR no lançamento.

Durante a etapa de configuração, seu arquivo docker será gerado com base no código da sua aplicação

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Using AWS region: {region}")

required_files = ['mcp_server.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            cognito_config['client_id']
        ],
        "discoveryUrl": cognito_config['discovery_url'],
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name=tool_name
)
print("Configuration completed ✓")

## Lançando Servidor MCP no AgentCore Runtime

Agora que temos um arquivo docker, vamos lançar o servidor MCP no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

## Armazenando Configuração para Acesso Remoto

Antes de podermos invocar nosso servidor MCP implantado, vamos armazenar o Agent ARN e a configuração do Cognito no AWS Systems Manager Parameter Store e AWS Secrets Manager para fácil recuperação:

In [ ]:
import boto3
import json

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='mcp_server/cognito/credentials',
        Description='Cognito credentials for MCP server',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='mcp_server/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials updated in Secrets Manager")

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

In [ ]:
import httpx

def stop_runtime_session_oauth(agent_arn, session_id, bearer_token, region):
    """Stop runtime session using OAuth bearer token"""
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/stopruntimesession"
    
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    body = {
        "runtimeSessionId": session_id,
        "agentRuntimeArn": agent_arn,
        "qualifier": "DEFAULT"
    }
    
    response = httpx.post(url, headers=headers, json=body, timeout=30.0)
    return response

print("✅ Helper function defined for OAuth session stopping")

### Demonstração do Ciclo de Vida da Sessão: Parando uma Sessão

Agora que o runtime está implantado, vamos demonstrar como parar uma sessão. Invocaremos o
servidor MCP com um ID de sessão personalizado e depois o pararemos usando autenticação OAuth.

In [ ]:
import uuid
import asyncio
from datetime import timedelta
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Generate custom session ID
demo1_session_id = str(uuid.uuid4())
print(f"📝 Demo 1 - Generated mcpSessionId: {demo1_session_id}")

# Prepare headers
encoded_arn = launch_result.agent_arn.replace(':', '%3A').replace('/', '%2F')
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

headers = {
    "authorization": f"Bearer {cognito_config['bearer_token']}",
    "Content-Type": "application/json",
    "Mcp-Session-Id": demo1_session_id
}

# Invoke MCP server
async def test_session():
    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"✅ Session active with {len(tools.tools)} tools")

await test_session()

# Stop the session using OAuth
print(f"\n🛑 Stopping session '{demo1_session_id}'...")
response = stop_runtime_session_oauth(
    launch_result.agent_arn,
    demo1_session_id,
    cognito_config['bearer_token'],
    region
)
print(f"✅ Session stopped (HTTP {response.status_code})")
print(f"   Response: {response.text}")
print(f"   MicroVM resources released")
print(f"💡 Runtime remains alive for new sessions")

# Note: You may see 'Session termination failed: 404' in the logs above.
# This is expected - it's the MCP client trying to auto-cleanup after we already stopped the session.
# The important part is the HTTP 200 response from our explicit stop_runtime_session_oauth call.

## Criando Cliente de Teste Remoto

Agora vamos criar um cliente para testar nosso servidor MCP implantado. Este cliente recuperará as credenciais necessárias da AWS e se conectará ao servidor implantado:

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
import base64
import time
from boto3.session import Session
from datetime import timedelta
import traceback

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_refresh_token(client_id, refresh_token, region):
    """Refresh access token using refresh token"""
    cognito_client = boto3.client('cognito-idp', region_name=region)
    auth_response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='REFRESH_TOKEN_AUTH',
        AuthParameters={'REFRESH_TOKEN': refresh_token}
    )
    return auth_response['AuthenticationResult']['AccessToken']

def get_valid_token(bearer_token, client_id, refresh_token, region):
    """Check token expiry and refresh if needed"""
    try:
        payload = bearer_token.split('.')[1]
        payload += '=' * (4 - len(payload) % 4)
        decoded = json.loads(base64.b64decode(payload))
        
        current_time = int(time.time())
        if decoded['exp'] - current_time < 300:
            print("🔄 Token expiring soon, refreshing...")
            new_token = get_refresh_token(client_id, refresh_token, region)
            print("✓ Token refreshed successfully")
            return new_token
        
        return bearer_token
    except Exception as e:
        print("🔄 Invalid token, refreshing...", e)
        traceback.print_exc()
        return get_refresh_token(client_id, refresh_token, region)

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        refresh_token = parsed_secret['refresh_token']
        client_id = parsed_secret['client_id']
        print("✓ Retrieved credentials from Secrets Manager")
        
        # Validate and refresh token if needed
        bearer_token = get_valid_token(bearer_token, client_id, refresh_token, region)
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: AGENT_ARN or BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Testando Seu Servidor MCP Implantado

Vamos testar nosso servidor MCP implantado usando o cliente remoto:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

## Invocando Ferramentas MCP Remotamente

Agora vamos criar um cliente aprimorado que não apenas lista ferramentas, mas também as invoca para demonstrar a funcionalidade completa do MCP:

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
import base64
import time
import uuid
import httpx
from boto3.session import Session
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_refresh_token(client_id, refresh_token, region):
    """Refresh access token using refresh token"""
    cognito_client = boto3.client('cognito-idp', region_name=region)
    auth_response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='REFRESH_TOKEN_AUTH',
        AuthParameters={'REFRESH_TOKEN': refresh_token}
    )
    return auth_response['AuthenticationResult']['AccessToken']

def get_valid_token(bearer_token, client_id, refresh_token, region):
    """Check token expiry and refresh if needed"""
    try:
        payload = bearer_token.split('.')[1]
        payload += '=' * (4 - len(payload) % 4)
        decoded = json.loads(base64.b64decode(payload))
        
        current_time = int(time.time())
        if decoded['exp'] - current_time < 300:
            print("🔄 Token expiring soon, refreshing...")
            new_token = get_refresh_token(client_id, refresh_token, region)
            print("✓ Token refreshed successfully")
            return new_token
        
        return bearer_token
    except:
        print("🔄 Invalid token, refreshing...")
        return get_refresh_token(client_id, refresh_token, region)

def stop_runtime_session_oauth(agent_arn, session_id, bearer_token, region):
    """Stop runtime session using OAuth bearer token via HTTP POST"""
    # Encode the ARN for URL path
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/stopruntimesession"
    
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    body = {
        "runtimeSessionId": session_id,
        "agentRuntimeArn": agent_arn,
        "qualifier": "DEFAULT"
    }
    
    response = httpx.post(url, headers=headers, json=body, timeout=30.0)
    return response

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        refresh_token = parsed_secret['refresh_token']
        client_id = parsed_secret['client_id']
        print("✓ Retrieved credentials from Secrets Manager")
        
        bearer_token = get_valid_token(bearer_token, client_id, refresh_token, region)
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    
    # Generate custom session ID
    mcp_session_id = str(uuid.uuid4())
    print(f"\n📝 Generated custom mcpSessionId: {mcp_session_id}")
    
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "Mcp-Session-Id": mcp_session_id
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream, write_stream, _
        ):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("✓ MCP session initialized")
                
                tool_result = await session.list_tools()
                print(f"\n📋 Found {len(tool_result.tools)} tools")
                
                # Test tools
                print("\n🧪 Testing tools...")
                add_result = await session.call_tool(name="add_numbers", arguments={"a": 5, "b": 3})
                print(f"   add_numbers(5, 3) = {add_result.content[0].text}")
                
                multiply_result = await session.call_tool(name="multiply_numbers", arguments={"a": 4, "b": 7})
                print(f"   multiply_numbers(4, 7) = {multiply_result.content[0].text}")
                
                greet_result = await session.call_tool(name="greet_user", arguments={"name": "Alice"})
                print(f"   greet_user('Alice') = {greet_result.content[0].text}")
                
                print("\n✅ MCP tool testing completed!")
        
        # Stop the session using OAuth bearer token
        print(f"\n🛑 Stopping session '{mcp_session_id}' (OAuth)...")
        response = stop_runtime_session_oauth(agent_arn, mcp_session_id, bearer_token, region)
        
        if response.status_code == 200:
            print(f"✅ Session stopped — microVM resources released")
            print(f"💡 Runtime remains alive for new sessions")
        else:
            print(f"⚠️  Status {response.status_code}: {response.text}")
                
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    asyncio.run(main())

## Testar Invocação de Ferramentas

Vamos testar nossas ferramentas MCP invocando-as de fato:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

### Demonstração do Ciclo de Vida da Sessão: Parando Entre Testes

Vamos demonstrar como parar uma sessão entre diferentes abordagens de teste.

In [ ]:
# Create and stop another session to show the pattern
demo2_session_id = str(uuid.uuid4())
print(f"📝 Demo 2 - Generated mcpSessionId: {demo2_session_id}")

headers2 = {
    "authorization": f"Bearer {cognito_config['bearer_token']}",
    "Content-Type": "application/json",
    "Mcp-Session-Id": demo2_session_id
}

async def test_session2():
    async with streamablehttp_client(mcp_url, headers2, timeout=timedelta(seconds=120), terminate_on_close=False) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print(f"✅ Session {demo2_session_id} created")

await test_session2()

print(f"🛑 Stopping session '{demo2_session_id}'...")
response = stop_runtime_session_oauth(
    launch_result.agent_arn,
    demo2_session_id,
    cognito_config['bearer_token'],
    region
)
print(f"✅ Session stopped (HTTP {response.status_code})")

## Próximos Passos

Agora que você implantou com sucesso um servidor MCP no AgentCore Runtime, você pode:

1. **Adicionar Mais Ferramentas**: Estenda seu servidor MCP com ferramentas adicionais
2. **Autenticação Personalizada**: Implemente autorizadores JWT personalizados
3. **Integração**: Integre com outros serviços do AgentCore

## Melhores Práticas do Ciclo de Vida da Sessão

Os custos do AgentCore Runtime são baseados em vCPU e Memória. Uma melhor prática para evitar custos indesejados é parar explicitamente a sessão ou configurar um timeout de inatividade apropriado, para que a sessão seja encerrada.

Para gerenciar custos de forma eficaz:

- **Configure timeout de inatividade**: Defina um timeout de inatividade apropriado durante a criação da sessão para parar automaticamente sessões inativas. Escolha um valor baseado no seu caso de uso (por exemplo, menor para desenvolvimento/teste, maior para cargas de trabalho de produção).
- **Pare sessões quando terminar**: Use `stop_runtime_session` para liberar os recursos do microVM para uma sessão específica enquanto mantém o runtime ativo para novas sessões.

## Limpeza

Agora vamos limpar o AgentCore Runtime e recursos associados. Deletamos o runtime primeiro para evitar custos indesejados, depois limpamos recursos de suporte como repositórios ECR e secrets do Secrets Manager.

In [ ]:
# --- Cleanup Resources ---
import boto3

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Step 1: Delete Secrets Manager secret first to minimize credential exposure window
try:
    secrets_client.delete_secret(
        SecretId='mcp_server/cognito/credentials',
        ForceDeleteWithoutRecovery=True
    )
    print("✅ Secrets Manager secret deleted")
except secrets_client.exceptions.ResourceNotFoundException:
    print("ℹ️  Secrets Manager secret not found")

# Step 2: Delete Parameter Store parameter
try:
    ssm_client.delete_parameter(Name='/mcp_server/runtime/agent_arn')
    print("✅ Parameter Store parameter deleted")
except ssm_client.exceptions.ParameterNotFound:
    print("ℹ️  Parameter Store parameter not found")

# Step 3: Delete the agent runtime to stop incurring costs
# AgentCore Runtime costs are based on vCPU and Memory
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Agent runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete agent runtime: {e}")

# Step 4: Delete the ECR repository
try:
    ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

print("\n✅ Cleanup completed successfully!")

# Parabéns!

Você completou com sucesso:

✅ **Criou um servidor MCP** com ferramentas personalizadas  
✅ **Testou localmente** usando cliente MCP  
✅ **Configurou autenticação** com Amazon Cognito  
✅ **Implantou na AWS** usando AgentCore Runtime  
✅ **Invocou remotamente** com autenticação adequada  
✅ **Aprendeu conceitos MCP** e melhores práticas  

Seu servidor MCP agora está rodando no Amazon Bedrock AgentCore Runtime e pronto para uso em produção!

## Resumo

Neste tutorial, você aprendeu como:
- Construir servidores MCP usando FastMCP
- Configurar transporte HTTP stateless para compatibilidade com AgentCore
- Configurar autenticação JWT com Amazon Cognito
- Implantar e gerenciar servidores MCP na AWS
- Testar tanto localmente quanto remotamente
- Usar clientes MCP para invocação de ferramentas

O servidor MCP implantado agora pode ser integrado em aplicações e fluxos de trabalho de IA maiores!